# ST 554 Homework 9
*By: Cass Crews*

## Introduction

The goal of this notebook is to practice building `MLlib` data pipelines. To do so, we will build pipelines for three separate model classes to tune the models on a training set and then compare the best models from each class using a test set. 

The [dataset](https://archive-beta.ics.uci.edu/dataset/73/mushroom) we will use provides mushroom characteristics for 8124 individual mushrooms. This dataset is available in the UCI Machine Learning Repository. The response variable will be an indicator of whether the mushroom is poisonous or edible. 

## Reading in Modules and Splitting Data

Before we begin working with the dataset, we need to read in the modules and sub-modules we'll need. We'll also need to initiate a `spark` instance. 

In [1]:
# Reading in modules and sub-modules, also initiating spark sequence
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler, \
    VectorIndexer, OneHotEncoder, Interaction
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Preventing warnings from printing
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 15:25:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


We are now ready to read in the data. Unfortunately, the UCI Repository server is down, so we'll need to read the data in from the local environment. 

As `pandas` provides simpler read-in capabilities, we'll read the dataset in as a `pandas` dataframe. We'll print the first few rows to ensure things read in correctly. 

In [2]:
mushroom_data = pd.read_csv("agaricus-lepiota.data", header = None, names = ["edible", "cap_shape", 
                                                                              "cap_surface", "cap_color", 
                                                                              "bruises", "odor", "gill_attach", 
                                                                              "gill_spacing", "gill_size", 
                                                                              "gill_color", "stalk_shape", "stalk_root", 
                                                                              "stalk_surface_ar", "stalk_surface_br", 
                                                                              "stalk_color_ar", "stalk_color_br", 
                                                                              "veil_type", "veil_color", "ring_number", 
                                                                              "ring_type", "spore_color", 
                                                                              "population", "habitat"])

mushroom_data.head()

,edible,cap_shape,cap_surface,cap_color,bruises,odor,gill_attach,gill_spacing,gill_size,gill_color,...,stalk_surface_br,stalk_color_ar,stalk_color_br,veil_type,veil_color,ring_number,ring_type,spore_color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


That's a lot of categorical variables! In fact, all the variables are categorical variables:

In [3]:
# Capturing data types and non-null counts
mushroom_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   edible            8124 non-null   object
 1   cap_shape         8124 non-null   object
 2   cap_surface       8124 non-null   object
 3   cap_color         8124 non-null   object
 4   bruises           8124 non-null   object
 5   odor              8124 non-null   object
 6   gill_attach       8124 non-null   object
 7   gill_spacing      8124 non-null   object
 8   gill_size         8124 non-null   object
 9   gill_color        8124 non-null   object
 10  stalk_shape       8124 non-null   object
 11  stalk_root        8124 non-null   object
 12  stalk_surface_ar  8124 non-null   object
 13  stalk_surface_br  8124 non-null   object
 14  stalk_color_ar    8124 non-null   object
 15  stalk_color_br    8124 non-null   object
 16  veil_type         8124 non-null   object
 17  veil_color    

The `object` type for each column indicates it is a categorical variable. Thus, the entire dataset is comprised of categorical variables we will need to convert to indicator variables for any non-tree-based models. 

Overall, it does seem the dataset was read in correctly, so let's store it in a `pyspark` SQL dataframe to utilize the `MLlib` functionality. We'll print out the first few rows and columns of this new dataframe to ensure the translation worked. 

In [4]:
mushroom_data_2 = spark.createDataFrame(mushroom_data)
mushroom_data_2.select("edible", "cap_shape", "cap_surface", 
                       "cap_color", "bruises", "odor", "gill_attach").show(5)

+------+---------+-----------+---------+-------+----+-----------+
|edible|cap_shape|cap_surface|cap_color|bruises|odor|gill_attach|
+------+---------+-----------+---------+-------+----+-----------+
|     p|        x|          s|        n|      t|   p|          f|
|     e|        x|          s|        y|      t|   a|          f|
|     e|        b|          s|        w|      t|   l|          f|
|     p|        x|          y|        w|      t|   p|          f|
|     e|        x|          s|        g|      f|   n|          f|
+------+---------+-----------+---------+-------+----+-----------+
only showing top 5 rows


This looks good. 

We are now ready to split the data into training and test sets using the `randomSplit()` method. We'll set aside 20\% of observations for the test set and confirm that actually occurs by printing the observation counts in the two dataframes. 

In [5]:
# Capturing training and test sets
train, test = mushroom_data_2.randomSplit([0.8,0.2], seed = 5)

# Printing observation counts for two dataframes
print(train.count(), test.count())

6489 1635


We have successfully split the full dataset into training and test sets! 

## Discussing Model Metric and Model Classes

### The Model Metric

We need to specify a model metric we will use to evaluate each model when tuning the model classes as well as to evaluate the best model of each class when predicting on the test set. As this is a classification problem, a natural choice is log-loss. It provides a more thorough evaluation of model performance than a more coarse metric such as accuracy. To understand this, let's consider the form of log-loss. In particular, let's consider mushroom edibility to be a binary random variable, $y$, with a value of 1 when a mushroom is edible and a value of 0 when it is not. A candidate model's log-loss for a given observation, $y_i$, and corresponding prediction, $p_i$, is, 

$$\text{log-loss} = -(y_i ln(\hat{p}_i) + (1-y_i) ln(1 - \hat{p}_i)).$$

Note that this metric will be close to 0 when our model correctly predicts mushroom edibility with confidence. Meanwhile, the metric will be larger if the model correctly classifies the obervation with low confidence or incorrectly classifies the observation. Thus, it accounts for both prediction accuracy and prediction confidence. When evaluating a model across multiple unseen observations, we will average the individual log-loss values. We will then select the candidate model that minimizes this average log-loss, which we will simply refer to as log-loss from this point forward. 

### The Model Classes

We now need to determine the model classes we will consider. As this dataset includes only categorical variables, tree-based models are a natural choice: they seamlessly handle indicators and allow for complex relationships among them. Thus, we will consider the random forest model. This model constructs a "forest" of classification trees that are used to predict the response class using majority rule. You may be wondering how we construct 500 or 1,000 trees from a single training set. We do so by repeatedly taking bootstrap samples from the training set and using each sample to fit a single tree. For this model class, we often tune three different hyperparameters: 

- The depth of each tree, or the maximum number of splits allowed along a single branch
- The minimum "leaf" size; that is, the minimum number of training observations allowed in any terminal node of a tree
- The size of the random sample of features considered when creating a new split in a tree.

The last hyperparameter is a key aspect of the random forest, as it ensures that not every tree's first, and often most important, splits are made using the same dominant predictors. This ensures we are not effectively fitting the same tree across each bootstrap sample, which would be a pointless endeavor. 

We will also consider two regularized logistic regression model classes: the logistic regression model with L1 penalty and the logistic regression model with L2 penalty. Both models penalize a standard logistic regression model for the magnitude of the estimated coefficients to prevent the model from becoming too complex given a large number of predictors. They differ in the exact form of that penalty: the L1 penalty penalizes the sum of the absolute value of the coefficients, while the L2 penalty penalizes the sum of the squares of the coefficients. The former results in some variables' coefficients being driven exactly to 0, effectively performing variable selection. Meanwhile, the latter does not perform variable selection but does provide the benefit that it will effectively construct a weighted average effect of any highly colinear predictors, preserving the information in each rather than retaining only one colinear predictor. For both models, we will tune a hyperparameter that determines the degree of penalization. We'll call this hyperparameter $\lambda$ in both cases. 

As a reminder, the logistic regression model is a generalized linear model where the log-odds of the response variable being equal to 1 is linear in the model parameters rather than the response variable itself having a lienar relationship with the predictors. This ensures all resultant model probability predictions are bounded between 0 and 1. 

Now that we've discussed the model classes, let's use pipelines to tune each and select an overall best model! 

## Tuning the Model Classes

In this section, we will create a modeling pipeline for each model class and use that pipeline to select the best model via 5-fold cross validation. 

A quick note before we get started: We will subset the feature variables to only eight variables a person could reasonably be expected to discern about a mushroom with the assistance of a guiding document. We won't discuss the meaning of each, but they are: cap shape, cap color, odor, stalk shape, veil color, spore color, population, and habitat.

### Random Forest Model

We'll start with the random forest model class. As this is a tree-based model, the piped transformations will be a bit different from those used for the regularized logistic regression models we will work with later. This is because tree-based models do not require categorical predictors to be converted to indicator variables for each class. Instead, we can simply index the categorical predictors with corresponding numeric-type columns; `VectorAssembler` requires all features be stored as numeric regardless of their actual type. 

Let's discuss the transformations that are constructed below: 

- `response_indexer` creates our official response variable, `label`, by creating a numeric-type dummy variable from `edible`
- `feature_numerizer` creates numeric feature variables from our original categorical features of interest, which is required by `VectorAssembler`
- `assembler` compiles the numeric versions of the feature values into a single column, which is what `pyspark` machine learning models require
- `feature_indexer` indicates that all our features are categorical variables so they will be treated as such in the random forest model.


In [6]:
# Converting edible to a numeric-type dummy variable for assembly
response_indexer = StringIndexer(inputCol = "edible", outputCol = "label", handleInvalid = "keep")

# Creating corresponding numeric-type feature variables for our categorical predictors of interest
feature_numerizer = StringIndexer(inputCols = ["cap_shape", "cap_color", "odor", "stalk_shape",  
                                                                              "veil_color", "spore_color", 
                                                                              "population", "habitat"],
                                outputCols = ["cap_shape_num", "cap_color_num", "odor_num", "stalk_shape_num",  
                                                                              "veil_color_num", "spore_color_num", 
                                                                              "population_num", "habitat_num"],
                                handleInvalid = "keep")

# Assembling feature values in a single column for pyspark ML models
assembler = VectorAssembler(inputCols = ["cap_shape_num", "cap_color_num", "odor_num", "stalk_shape_num",  
                                                                              "veil_color_num", "spore_color_num", 
                                                                              "population_num", "habitat_num"],
                            outputCol = "features_num")

# Indicating all features are actually categorical for the random forest model
feature_indexer = VectorIndexer(inputCol = "features_num", outputCol = "features") 



Now that we've constructed all the transformations for the random forest model class, we simply need to specify the model class itself; construct a grid of candidate hyperparameter values, and construct our pipeline. Candidate hyperparameter values are:
- Maximum tree depth (`maxDepth`): 5, 10, 15, or 20 splits per branch
- Minimum leaf size (`minInstancesPerNode`): 3, 5, 25, or 100 training observations
- Number of randomly selected features per split (`featureSubsetStrategy`): 2 through 8 (total features)

Note that for each random forest model, we are constructing 500 trees, a common forest size. 

In [7]:
# Capturing our model, the random forest classification model with 500 trees
rf = RandomForestClassifier(numTrees = 500, seed = 5)

# Specifying parameter grid with candidate values of maxDepth, minInstancesPerNode (min leaf size),
# and featureSubsetStrategy (per-split feature sample size)
paramGrid = ParamGridBuilder() \
    .addGrid(rf.maxDepth, [5, 10, 15, 20]) \
    .addGrid(rf.minInstancesPerNode, [3, 5, 25, 100]) \
    .addGrid(rf.featureSubsetStrategy, ["2", "3", "4", "5", "6", "7", "8"]) \
    .build()

# Constructing our pipeline
pipeline = Pipeline(stages = [response_indexer, feature_numerizer, assembler, feature_indexer, rf])

We are now ready to construct the `CrossValidator` instance we will use to tune the random forest model. Note that we formally specify that our model metric is log-loss, indicate 5-fold cross validation, utilize three cores to speed up runtime, and set a seed for reproducibility. 

In [8]:
# Constructing CrossValidator instance
crossval = CrossValidator(estimator = pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = MulticlassClassificationEvaluator(metricName = "logLoss"), # specifying log-loss model metric
                          numFolds = 5, # 5-fold CV
                          parallelism = 3, # 3 cores
                          seed = 5) # setting seed for reproducibility

Let's "fit" the `CrossValidator` object to obtain the combination of hyperparameters that produces the lowest cross-validated log-loss. 

In [9]:
# Tuning random forest
rf_cv = crossval.fit(train)

We now have cross-validated log-loss values for all combinations of hyperparameters. Let's capture those metric values along with their corresponding hyperparameter values and extract the 10 best-performing combinations. 

In [10]:
# Creating empty list to append model metric-hyperparameter pairs to
tune_list = []

# Appending model metric-hyperparameter pairs
for i in range(len(paramGrid)):
    tune_list.append([rf_cv.avgMetrics[i], paramGrid[i].values()])

# Sorting the list by cross-validated log-loss
tune_list.sort(key = lambda x: x[0])

# Printing top 10 models
tune_list[0:10]

[[0.0007193584889329721, dict_values([15, 3, '8'])],
 [0.0007193584889329723, dict_values([20, 3, '8'])],
 [0.000722538368008981, dict_values([10, 3, '8'])],
 [0.000747747769004484, dict_values([15, 3, '7'])],
 [0.000747747769004484, dict_values([20, 3, '7'])],
 [0.0007619403954381576, dict_values([10, 3, '7'])],
 [0.000845235736135309, dict_values([15, 3, '6'])],
 [0.0008452357361353092, dict_values([20, 3, '6'])],
 [0.0009040406547522958, dict_values([10, 3, '6'])],
 [0.0010817358994400894, dict_values([20, 3, '5'])]]

We have a tie! The two best-performing models have a minimum leaf size of 3 and use all features when determining which feature to split on next. However, one has a depth of 15 splits, while the other has a tree depth of 20 splits. These are very similar models, but it is still interesting that their cross-validated log-loss values are identical. As the model with a depth of 15 splits will occur first in the hyperparameter grid, it will be considered the best model by `CrossValidator`. 

### Logistic Regression Model with L1 Penalty

We are now ready to construct the pipeline for the logistic regression model with L1 Penalty. We will use the same eight features we considered in the random forest case. As we previously alluded to, we will need to convert each feature into corresponding indicator variables as logistic regression models cannot handle multi-level categorical variables as flexibly as tree-based models. However, we can still use the `response_indexer` and `feature_numerizer` transformations from above as we need to repeat those steps in this pipeline.  Additionally, we need the following transformations:

- `encode_features`: This transformation actually converts the categorical features to corresponding (correctly sparse) indicators for each level of each feature; note that the resultant columns are actually columns of vectors that compile seamlessly when we use `VectorAssember`, precluding any requirement to create a column for each indicator
- `lr_assember`: This step is simply a feature column assembly transformation, conceptually the same as `assembler` above.

In [11]:
# Creating sparse indicator sets from the categorical features
encode_features = OneHotEncoder(inputCols = ["cap_shape_num", "cap_color_num", "odor_num", "stalk_shape_num",  
                                                                              "veil_color_num", "spore_color_num", 
                                                                              "population_num", "habitat_num"],
                                outputCols = ["cap_shape_enc", "cap_color_enc", "odor_enc", "stalk_shape_enc",  
                                                                              "veil_color_enc", "spore_color_enc", 
                                                                              "population_enc", "habitat_enc"],
                                handleInvalid = "keep")

# Compiling the feature indicators into a single
lr_assembler = VectorAssembler(inputCols = ["cap_shape_enc", "cap_color_enc", "odor_enc", "stalk_shape_enc",  
                                                                              "veil_color_enc", "spore_color_enc", 
                                                                              "population_enc", "habitat_enc"],
                              outputCol = "features")

We now have all the transformations we need to construct our pipeline. Let's create a logistic regression object that will be our final pipeline piece, specify a grid of candidate $\lambda$ values, and construct the pipeline. For $\lambda$, we will consider 25 values between $10^{-6}$ and $10^2$. Note that we indicate we want an L1 penalty by specifying `elasticNetParam = 1.0` within `LogisticRegression()`. 

In [12]:
# Creating LogisticRegression object 
lr_l1 = LogisticRegression(elasticNetParam = 1.0)

# Specifying a sequence of candidate lambda values
l1_paramGrid = ParamGridBuilder() \
    .addGrid(lr_l1.regParam, np.logspace(-6, 2, 25)).build()

# Constructing L1-penalized logistic regression pipeline
lr_l1_pipeline = Pipeline(stages = [response_indexer, feature_numerizer, encode_features, lr_assembler, lr_l1])

Now that we've constructed the pipeline and the candidate hyperparameter grid, let's pass the pipeline and grid to `CrossValidator()` so we can use 5-fold CV to identify the best $\lambda$ value. We'll again formally specify that we want to select the best model using log-loss. We'll also utilize three virtual cores to speed up runtime and set a seed for reproducibility. 

In [13]:
# Constructing CrossValidator object to tune the logistic regression model with L1 penalty
lr_l1_crossval = CrossValidator(estimator = lr_l1_pipeline,
                                estimatorParamMaps = l1_paramGrid,
                                evaluator = MulticlassClassificationEvaluator(metricName = "logLoss"), # specifying log-loss model metric
                                numFolds = 5, # 5-fold CV
                                parallelism = 3, # 3 cores
                                seed = 5)

We are now ready to tune the logistic regression model with L1 penalty! 

In [14]:
# Using 5-fold CV to tune the model
lr_l1_cv = lr_l1_crossval.fit(train)

We now have five log-loss values for each of the 25 candidate $\lambda$ values. Just as we did for the random forest above. Let's pair the $\lambda$ values with their corresponding average log-loss to determine the best model for predicting edibility. 

In [15]:
# Creating empty list to append model metric-hyperparameter pairs to
lr_l1_tune_list = []

# Appending model metric-hyperparameter pairs
for i in range(len(l1_paramGrid)):
    lr_l1_tune_list.append([lr_l1_cv.avgMetrics[i], l1_paramGrid[i].values()])

# Sorting the list by cross-validated log-loss
lr_l1_tune_list.sort(key = lambda x: x[0])

# Printing top 10 models
lr_l1_tune_list[0:10]

[[7.890248031358858e-06, dict_values([1e-06])],
 [1.6308412601930365e-05, dict_values([2.1544346900318822e-06])],
 [3.719775290015087e-05, dict_values([4.641588833612782e-06])],
 [8.013218469800586e-05, dict_values([1e-05])],
 [0.00017637751419110343, dict_values([2.1544346900318823e-05])],
 [0.00037175936786815785, dict_values([4.641588833612772e-05])],
 [0.000794059893068365, dict_values([0.0001])],
 [0.0016983976932895138, dict_values([0.00021544346900318823])],
 [0.0035382749033764566, dict_values([0.00046415888336127773])],
 [0.007188895725182054, dict_values([0.001])]]

Interestingly, the best model has the smallest $\lambda$ value among the candidates, meaning the best-performing model is the one that penalizes the coefficients the least and produces results similar to that of an unregularized logistic regression model. Also, these cross-validated log-loss values are much lower than those produced by the top-performing random forest models. I'm curious whether this differential will hold when we use each top-performing model from these classes to predict for the test set! 

## Logistic Regression Model with L2 Penalty

We are now ready to construct a pipeline to tune our final model class, the logistic regression model with L2 penalty. Given that the best-performing L1-penalized model effectively does not penalize coefficient size at all, we can expect a similar result for this model class. As we will use the same eight features we used for the other model classes and this model class requires the same form for the features as the other regularized logistic regression model, we don't need to construct any new transformers. Indeed, the only change we need to make to the L1-penalized pipeline is the estimator itself. 

We'll make that change now. We'll also construct the grid for $\lambda$, which will include the same candidate values we considered in the L1-penalty case: 25 values ranging from $10^{-6}$ to $10^2$. Note that while setting `elasticNetParam` to 1 resulted in an L1 penalty, setting this function parameter to 0 results in an L2 penalty. 



In [16]:
# Creating LogisticRegression object 
lr_l2 = LogisticRegression(elasticNetParam = 0.0)

# Specifying a sequence of candidate lambda values
l2_paramGrid = ParamGridBuilder() \
    .addGrid(lr_l2.regParam, np.logspace(-6, 2, 25)).build()

# Constructing L2-penalized logistic regression pipeline
lr_l2_pipeline = Pipeline(stages = [response_indexer, feature_numerizer, encode_features, lr_assembler, lr_l2])

Now that we've constructed the pipeline, we will construct a new `CrossValidator()` object with the same non-pipeline inputs we used in the L1 case; the parameter grid is implicitly the same. 

In [17]:
# Constructing CrossValidator object to tune the logistic regression model with L2 penalty
lr_l2_crossval = CrossValidator(estimator = lr_l2_pipeline,
                                estimatorParamMaps = l2_paramGrid,
                                evaluator = MulticlassClassificationEvaluator(metricName = "logLoss"), # specifying log-loss model metric
                                numFolds = 5, # 5-fold CV
                                parallelism = 3, # 3 cores
                                seed = 5)

Let's tune this model class! 

In [18]:
# Tuning the L2-penalized logistic regression model using 5-fold CV
lr_l2_cv = lr_l2_crossval.fit(train)

We again need to extract the average cross-validated log-loss for each model to determine which value of $\lambda$ produces the best predictive model of edibility. 

In [19]:
# Creating empty list to append model metric-hyperparameter pairs to
lr_l2_tune_list = []

# Appending model metric-hyperparameter pairs
for i in range(len(l2_paramGrid)):
    lr_l2_tune_list.append([lr_l2_cv.avgMetrics[i], l2_paramGrid[i].values()])

# Sorting the list by cross-validated log-loss
lr_l2_tune_list.sort(key = lambda x: x[0])

# Printing top 10 models
lr_l2_tune_list[0:10]

[[2.2888069427333794e-05, dict_values([1e-06])],
 [4.443732621785919e-05, dict_values([2.1544346900318822e-06])],
 [8.528628891485199e-05, dict_values([4.641588833612782e-06])],
 [0.00016170474213935183, dict_values([1e-05])],
 [0.0003016188470969827, dict_values([2.1544346900318823e-05])],
 [0.0005545556359782442, dict_values([4.641588833612772e-05])],
 [0.000999636988883894, dict_values([0.0001])],
 [0.001758436820165114, dict_values([0.00021544346900318823])],
 [0.0030033995406249527, dict_values([0.00046415888336127773])],
 [0.004998666445513369, dict_values([0.001])]]

Just as we suspected, it was again the case that the least penalized model performed best with regard to minimizing log-loss. We have again found ourselves with a "best" model that is effectively the unregularized logistic regression model. 

Overall, the best-performing models from each class produce miniscule cross-validated log-loss values! 

## Selecting a Final Model

We now have the best-performing model from each of the three model classes. We are ready to predict edibility for the test set to determine our best overall model. We will again use log-loss to make our decision. 

Programmatically, this step is very straightforward. Now that we have "fit" each `CrossValidator` object, we simply need to use these objects to "transform" the test set and calculate the corresponding log-loss. 

In [20]:
# Calculating test log-loss for each "best" model
rf_test_logloss = MulticlassClassificationEvaluator(metricName = "logLoss") \
    .evaluate(rf_cv.transform(test)) #random forest case
lr_l1_test_logloss = MulticlassClassificationEvaluator(metricName = "logLoss") \
    .evaluate(lr_l1_cv.transform(test)) #LR L1 case
lr_l2_test_logloss = MulticlassClassificationEvaluator(metricName = "logLoss") \
    .evaluate(lr_l2_cv.transform(test)) #LR L2 case

# Printing test log-loss values
print("Random Forest Log-Loss:", rf_test_logloss,
      "\nL1-Penalized Logistic Regression Log-Loss:", lr_l1_test_logloss,
      "\nL2-Penalized Logistic Regression Log-Loss:", lr_l2_test_logloss)

Random Forest Log-Loss: 0.00030903668741383576 
L1-Penalized Logistic Regression Log-Loss: 6.373291044898313e-06 
L2-Penalized Logistic Regression Log-Loss: 1.4590161623185014e-05


The logistic regression models produce substantially smaller test log-loss values. However, all three models produce values near 0. It seems our features are very effective for predicting edibility, meaning it's not necessary to deploy a complex model such as the random forest. 

As we are using log-loss to select the best overall model, we will choose the logistic regression model with an L1 penalty and $\lambda=0.000001$ as our final model. Of course, we would want to fit this specific model to the full dataset before deploying it. 

A note: one could argue that the better approach is to use our results so far as an indication that the L1-penalized logistic regression is the best overall model class and then repeat the entire cross validation tuning process for this class on the full dataset. This is because we have effectively shown the full tuning procedure for the L1-penalized logistic regression model produces the best out-of-sample performance by using the results of the tuning procedure to predict for the test set.

Another note: One could also argue, based on the fact that the best regularized logistic regression models were the models with the smallest possible penalization, that we should also consider a non-regularized logistic regression model. However, that is beyond the scope of this notebook. 

Because all three "best" models produce such small test log-loss values, let's see what their test accuracy values are. 

In [21]:
# Calculating test accuracy for each "best" model
rf_test_acc = MulticlassClassificationEvaluator(metricName = "accuracy") \
    .evaluate(rf_cv.transform(test)) #random forest case
lr_l1_test_acc = MulticlassClassificationEvaluator(metricName = "accuracy") \
    .evaluate(lr_l1_cv.transform(test)) #LR L1 case
lr_l2_test_acc = MulticlassClassificationEvaluator(metricName = "accuracy") \
    .evaluate(lr_l2_cv.transform(test)) #LR L2 case

# Printing test accuracy values
print("Random Forest Accuracy:", rf_test_acc,
      "\nL1-Penalized Logistic Regression Accuracy:", lr_l1_test_acc,
      "\nL2-Penalized Logistic Regression Accuracy:", lr_l2_test_acc)

Random Forest Accuracy: 1.0 
L1-Penalized Logistic Regression Accuracy: 1.0 
L2-Penalized Logistic Regression Accuracy: 1.0


Fascinating! All three models produce perfect test accuracy. However, if we dig into the discussion boards for this dataset, we will find that this is not unusual at all. It is actually relatively easy to separate edible mushrooms from poisonous ones using machine learning models! 